# Serveur Colab : LLM (Qwen) + VLM + CLIP

Ce notebook lance sur le GPU de Colab, derrière des tunnels HTTPS (cloudflared, sans compte) :
- **Ollama** : `qwen2.5:7b` (cerveau de l'agent) et `qwen2.5vl:7b` (description des images) ;
- **Serveur d'embeddings CLIP** ViT-B-32 : le notebook RAG sur ton PC lui envoie textes et images et reçoit les vecteurs normalisés.

**Avant de lancer :** Exécution → Modifier le type d'exécution → GPU **T4**.

⚠️ Les URL changent à chaque session ; la session gratuite s'arrête après inactivité : laisse l'onglet ouvert.

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print(gpu or "AUCUN GPU")
if not gpu:
    raise RuntimeError("Pas de GPU : Exécution → Modifier le type d'exécution → T4 GPU, puis relancer depuis le début.")

## 1. Installer Ollama

In [ ]:
# pciutils + lshw : permettent au script d'installation de détecter le GPU (sinon : « Unable to detect NVIDIA/AMD GPU »)
!apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

## 2. Démarrer le serveur et télécharger les modèles
- `qwen2.5:7b` (≈4,7 Go) : cerveau de l'agent, très bon en *tool calling*.
- `qwen2.5vl:7b` (≈6 Go) : VLM qui génère les descriptions des images à l'indexation.
Ollama charge un modèle à la fois : le passage de l'un à l'autre prend quelques secondes.

In [ ]:
import os, subprocess, time, requests
MODEL = "qwen2.5:7b"       # cerveau de l'agent (appel d'outils)
VLM = "qwen2.5vl:7b"       # décrit les images à l'indexation
env = dict(os.environ, OLLAMA_HOST="0.0.0.0:11434", OLLAMA_KEEP_ALIVE="-1", OLLAMA_CONTEXT_LENGTH="8192", OLLAMA_MAX_LOADED_MODELS="1")  # 1 modèle en VRAM à la fois (T4 16 Go)
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2); break
    except Exception:
        time.sleep(1)
print("Serveur Ollama prêt")

# Ollama a-t-il trouvé le GPU ? (à vérifier AVANT de télécharger ~10 Go de modèles)
time.sleep(3)
log = open("ollama.log").read()
print("\n".join(l[-220:] for l in log.splitlines() if any(k in l.lower() for k in ("inference compute", "compatible gpu", "library=")))[-1200:] or "(pas d'info GPU dans ollama.log)")
if "no compatible gpu" in log.lower():
    raise RuntimeError("Ollama n'utilise pas le GPU (il tournerait sur CPU, beaucoup trop lent). "
                       "Vérifie le type d'exécution (T4) puis relance les cellules d'installation et de démarrage.")

!ollama pull $MODEL
!ollama pull $VLM

## 3. Ouvrir le tunnel public

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import re, subprocess, time, requests

def open_tunnel(port, name, health="/", wait=150):
    """Ouvre un tunnel public vers localhost:port et attend qu'il soit réellement joignable.
    L'URL apparaît dans les logs avant que le DNS soit propagé : on attend qu'elle réponde."""
    log = f"{name}.log"
    proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{port}", "--no-autoupdate"],
                            stdout=open(log, "w"), stderr=subprocess.STDOUT)
    url, t0 = None, time.time()
    while time.time() - t0 < 90 and not url:
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open(log).read())
        url = m.group(0) if m else None
        time.sleep(1)
    if not url:
        raise RuntimeError(f"Tunnel {name} : URL introuvable, voir {log}")
    time.sleep(8)   # laisse le DNS se propager avant le premier essai
    t0 = time.time()
    while time.time() - t0 < wait:
        try:
            if requests.get(url + health, timeout=10).status_code < 500:
                print(f"Tunnel {name} joignable : {url}")
                return proc, url
        except requests.RequestException:
            pass
        time.sleep(4)
    print(f"⚠️ Tunnel {name} créé mais pas encore joignable depuis Colab (DNS en cours de propagation) : {url}\n"
          "   Il fonctionnera sans doute depuis ton PC dans quelques minutes.")
    return proc, url

tunnel, url = open_tunnel(11434, "tunnel_ollama", "/")

## 4. Tester Qwen
Trois contrôles : GPU présent, chargement du modèle (1re fois : 30 à 90 s), puis appel d'outil. Chaque étape affiche sa durée.

In [ ]:
import json, subprocess, time, requests

gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
print(gpu if gpu else "⚠️ PAS DE GPU : Exécution → Modifier le type d'exécution → T4, puis relancer depuis le début.")

def chat(payload, base="http://localhost:11434", timeout=240):
    t = time.time()
    r = requests.post(f"{base}/api/chat", json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()["message"], time.time() - t

# 1) Chargement du modèle en mémoire GPU
msg, dt = chat({"model": MODEL, "stream": False, "options": {"num_predict": 5},
                "messages": [{"role": "user", "content": "Réponds seulement OK."}]})
print(f"1) Chargement + réponse : {dt:.0f} s -> {msg['content']!r}")
print(subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout)   # PROCESSOR doit afficher « 100% GPU »

# 2) Appel d'outil (en local sur Colab)
tools = [{"type": "function", "function": {"name": "meteo", "description": "Météo d'une ville",
          "parameters": {"type": "object", "properties": {"ville": {"type": "string"}}, "required": ["ville"]}}}]
msg, dt = chat({"model": MODEL, "stream": False, "tools": tools,
                "messages": [{"role": "user", "content": "Quelle est la météo à Rabat ?"}]})
print(f"2) Appel d'outil : {dt:.0f} s ->", json.dumps(msg.get("tool_calls"), ensure_ascii=False))

# 3) Même requête via le tunnel public (celui que verra ton PC)
try:
    r = requests.get(f"{url}/api/tags", timeout=30)
    print("3) Tunnel :", r.status_code, "-", [m["name"] for m in r.json()["models"]])
except Exception as exc:
    print("3) Tunnel pas encore joignable depuis Colab :", type(exc).__name__)

## 5. Serveur d'embeddings CLIP
Un petit serveur FastAPI (port 8000) charge CLIP ViT-B-32 sur le GPU et expose : `/embed/text`, `/embed/image` (vecteurs **normalisés**), `/chunk` (découpage du texte en fragments de 60 tokens CLIP) et `/health`. Il est protégé par une clé API (`x-api-key`) générée à chaque session.

In [ ]:
!pip -q install fastapi "uvicorn[standard]" open-clip-torch
import os, secrets, subprocess, sys, time, requests

SERVER_CODE = r'''
import base64, io, os
import open_clip, torch
from fastapi import FastAPI, Header, HTTPException
from PIL import Image
from pydantic import BaseModel

MODEL, PRETRAINED = "ViT-B-32", "laion2b_s34b_b79k"
API_KEY = os.environ["EMBED_API_KEY"]
device = "cuda" if torch.cuda.is_available() else "cpu"

model, _, preprocess = open_clip.create_model_and_transforms(MODEL, pretrained=PRETRAINED)
model = model.to(device).eval()
tokenizer = open_clip.get_tokenizer(MODEL)
bpe = open_clip.tokenizer.SimpleTokenizer()
app = FastAPI(title="Serveur d'embeddings CLIP")


def check(key):
    if key != API_KEY:
        raise HTTPException(status_code=401, detail="clé API invalide")


def normed(v):
    return (v / v.norm(dim=-1, keepdim=True).clamp_min(1e-12)).float().cpu().tolist()


class TextReq(BaseModel):
    texts: list[str]


class ImageReq(BaseModel):
    image_b64: str


class ChunkReq(BaseModel):
    text: str
    size: int = 60
    overlap: int = 10


@app.get("/health")
def health():
    return {"model": MODEL, "pretrained": PRETRAINED, "device": device, "dim": 512}


@app.post("/embed/text")
def embed_text(req: TextReq, x_api_key: str = Header(None)):
    check(x_api_key)
    with torch.inference_mode():
        tok = tokenizer(req.texts).to(device)  # tronque à 77 tokens
        return {"vectors": normed(model.encode_text(tok))}


@app.post("/embed/image")
def embed_image(req: ImageReq, x_api_key: str = Header(None)):
    check(x_api_key)
    im = Image.open(io.BytesIO(base64.b64decode(req.image_b64))).convert("RGB")
    with torch.inference_mode():
        x = preprocess(im).unsqueeze(0).to(device)
        return {"vector": normed(model.encode_image(x))[0]}


@app.post("/chunk")
def chunk(req: ChunkReq, x_api_key: str = Header(None)):
    """Découpe en fragments d'au plus `size` tokens CLIP, texte d'origine conservé."""
    check(x_api_key)
    if not 0 <= req.overlap < req.size:
        raise HTTPException(status_code=400, detail="overlap doit être < size")
    words = req.text.split()
    counts = [len(bpe.encode(w)) for w in words]
    chunks, start = [], 0
    while start < len(words):
        end, n = start, 0
        while end < len(words) and (n + counts[end] <= req.size or end == start):
            n += counts[end]
            end += 1
        chunks.append(" ".join(words[start:end]))
        if end >= len(words):
            break
        back, m = end, 0
        while back > start + 1 and m + counts[back - 1] <= req.overlap:
            back -= 1
            m += counts[back]
        start = max(back, start + 1)
    return {"chunks": chunks}
'''
open("clip_server.py", "w").write(SERVER_CODE)

EMBED_API_KEY = secrets.token_urlsafe(16)
embed_server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "clip_server:app", "--host", "0.0.0.0", "--port", "8000"],
    env=dict(os.environ, EMBED_API_KEY=EMBED_API_KEY), stdout=open("clip_server.log", "w"), stderr=subprocess.STDOUT)

for _ in range(180):   # 1er lancement : téléchargement des poids CLIP (~600 Mo)
    try:
        print("Serveur CLIP prêt :", requests.get("http://localhost:8000/health", timeout=2).json()); break
    except Exception:
        if embed_server.poll() is not None:
            raise RuntimeError("Le serveur CLIP s'est arrêté : voir clip_server.log")
        time.sleep(2)
else:
    raise TimeoutError("Serveur CLIP trop long à démarrer : voir clip_server.log")

### Tunnel public et test

In [ ]:
embed_tunnel, embed_url = open_tunnel(8000, "tunnel_embed", "/health")
h = {"x-api-key": EMBED_API_KEY}
try:   # simple vérification : un échec ici (DNS en retard) n'empêche pas d'utiliser le serveur depuis le PC
    v = requests.post(f"{embed_url}/embed/text", json={"texts": ["centrifugal pump"]}, headers=h, timeout=60).json()["vectors"][0]
    print("dim =", len(v), "| norme =", round(sum(x * x for x in v) ** 0.5, 4))
    print(requests.post(f"{embed_url}/chunk", json={"text": "word " * 200}, headers=h, timeout=60).json()["chunks"][0][:60], "…")
except Exception as exc:
    print("⚠️ Vérification depuis Colab impossible pour l'instant :", type(exc).__name__, "- réessaie cette cellule dans 1 minute.")

print("\n=== À copier dans le fichier .env de ton PC ===")
print(f"OLLAMA_BASE_URL={url}")
print(f"LLM_MODEL={MODEL}")
print(f"VLM_MODEL={VLM}")
print(f"EMBED_BASE_URL={embed_url}")
print(f"EMBED_API_KEY={EMBED_API_KEY}")

## 6. Garder la session active
Laisse cette cellule tourner tant que tu utilises le RAG (Interrompre pour arrêter).

In [ ]:
import time
while True:
    time.sleep(60)
    procs = {"ollama": server, "tunnel ollama": tunnel, "serveur CLIP": embed_server, "tunnel CLIP": embed_tunnel}
    arretes = [n for n, p in procs.items() if p.poll() is not None]
    if arretes:
        print("⚠️ Arrêtés :", arretes, "→ relance les cellules correspondantes."); break